In [ ]:
# pip install geopandas pandas planetary-computer pystac-client requests shapely

from pathlib import Path
import json
import os
import time

import geopandas as gpd
import pandas as pd
import planetary_computer
import pystac_client
import requests

from shapely.geometry import shape


sentinel2_grid_kml = "S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml" #from : https://sentiwiki.copernicus.eu/web/s2-documents
time_range = "2020-01-01T00:00:00Z/2021-12-31T23:59:59Z" #date
collection_name = "sentinel-1-grd"


out_root = Path("/media/sara/T7/s1_grd_monthly_2020_2021_best_tiles") 


best_tiles = [
    "21JXJ",
    "24LTR",
    "22KGB",
    "23LLG",
] 

min_tile_coverage = 0.65  #The fraction of the tile covered by the orbit 1=100%


area_crs = "EPSG:6933"


def pick_tile_id_column(gdf):
    candidates = ["Name", "name", "tile", "tile_id", "TILE_ID", "mgrs", "MGRS"]
    for c in candidates:
        if c in gdf.columns:
            return c
    raise ValueError(f"Could not find tile ID column. Available columns: {list(gdf.columns)}")

def month_start(ts: pd.Timestamp) -> pd.Timestamp:
    return pd.Timestamp(year=ts.year, month=ts.month, day=1)

def ensure_writable_directory(path: Path):
    path.mkdir(parents=True, exist_ok=True)

    test_file = path / ".write_test.tmp"
    try:
        with open(test_file, "w") as f:
            f.write("ok")
        test_file.unlink()
    except Exception as e:
        raise RuntimeError(f"Output directory is not writable: {path}\nError: {e}")

def download_file(
    url: str,
    dst: Path,
    chunk_size: int = 1024 * 256,
    connect_timeout: int = 30,
    read_timeout: int = 600,
):
    tmp_dst = Path(str(dst) + ".part")

    with requests.get(url, stream=True, timeout=(connect_timeout, read_timeout)) as r:
        r.raise_for_status()

        with open(tmp_dst, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)

    os.replace(tmp_dst, dst)

def download_asset_with_fresh_signature(item, band: str, dst: Path, retries: int = 5):
    asset = item.assets.get(band)
    if asset is None:
        return

    if dst.exists() and dst.stat().st_size > 0:
        print(f"  skipping existing {dst.name}")
        return

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            signed_asset = planetary_computer.sign(asset)
            download_file(signed_asset.href, dst)
            return

        except requests.exceptions.RequestException as e:
            last_error = e

            part_file = Path(str(dst) + ".part")
            if part_file.exists():
                part_file.unlink()

            print(f"    attempt {attempt}/{retries} failed for {band.upper()}: {e}")

            if attempt < retries:
                wait_s = 5 * attempt
                print(f"    waiting {wait_s}s and retrying...")
                time.sleep(wait_s)

    raise last_error


print(f"Using output directory: {out_root}")
ensure_writable_directory(out_root)


tiles_gdf = gpd.read_file(sentinel2_grid_kml)

if tiles_gdf.crs is None:
    raise ValueError("Tile grid has no CRS")

tiles_gdf = tiles_gdf.to_crs(epsg=4326)
tile_id_col = pick_tile_id_column(tiles_gdf)

tiles_gdf[tile_id_col] = tiles_gdf[tile_id_col].astype(str).str.strip()

selected_tiles = tiles_gdf[tiles_gdf[tile_id_col].isin(best_tiles)].copy()

missing_tiles = sorted(set(best_tiles) - set(selected_tiles[tile_id_col]))
if missing_tiles:
    raise ValueError(f"These tiles were not found in the KML: {missing_tiles}")

selected_tiles_area = selected_tiles.to_crs(area_crs).copy()

print("Tiles found:")
print(selected_tiles[[tile_id_col]])


catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)


selected_rows = []

for idx, row in selected_tiles.iterrows():
    tile_name = row[tile_id_col]
    tile_geom_wgs84 = row.geometry
    tile_geom_area = selected_tiles_area.loc[idx].geometry
    tile_area = tile_geom_area.area

    print(f"\nSearching {tile_name} ...")

    search = catalog.search(
        collections=[collection_name],
        intersects=tile_geom_wgs84.__geo_interface__,
        datetime=time_range,
    )

    items = sorted(search.item_collection(), key=lambda item: item.datetime)
    print(f"Found {len(items)} GRD items")

    records = []

    for item in items:
        item_geom_wgs84 = shape(item.geometry)

        if not item_geom_wgs84.intersects(tile_geom_wgs84):
            continue

        item_geom_area = gpd.GeoSeries([item_geom_wgs84], crs="EPSG:4326").to_crs(area_crs).iloc[0]

        inter_area = item_geom_area.intersection(tile_geom_area).area
        if inter_area <= 0:
            continue

        coverage_ratio = inter_area / tile_area

        if coverage_ratio < min_tile_coverage:
            continue

        dt = pd.to_datetime(item.datetime, utc=True).tz_localize(None)

        has_vv = "vv" in item.assets
        has_vh = "vh" in item.assets
        has_both = has_vv and has_vh

        records.append(
            {
                "tile": tile_name,
                "month": month_start(dt),
                "datetime": dt,
                "item_id": item.id,
                "has_vv": has_vv,
                "has_vh": has_vh,
                "has_both": has_both,
                "coverage_ratio": coverage_ratio,
                "item": item,
            }
        )

    if not records:
        print(f"No scenes found for {tile_name} after coverage filter")
        continue

    df = pd.DataFrame(records)

    df_selected = (
        df.sort_values(
            ["month", "coverage_ratio", "has_both", "datetime"],
            ascending=[True, False, False, True]
        )
        .drop_duplicates(subset=["month"], keep="first")
        .sort_values("month")
        .reset_index(drop=True)
    )

    print(f"Selected {len(df_selected)} monthly scenes for {tile_name}")
    print(df_selected[["month", "item_id", "coverage_ratio", "has_vv", "has_vh"]])

    selected_rows.append(df_selected)

if not selected_rows:
    raise ValueError("No monthly scenes found for any tile. Try lowering min_tile_coverage.")

df_all = pd.concat(selected_rows, ignore_index=True).sort_values(["tile", "month"])

print("\nFinal monthly selection:")
print(
    df_all[["tile", "month", "datetime", "item_id", "coverage_ratio", "has_vv", "has_vh"]]
    .to_string(index=False)
)

selection_csv = out_root / "selected_monthly_scenes.csv"
df_all.drop(columns=["item"]).to_csv(selection_csv, index=False)
print(f"\nSaved selection CSV: {selection_csv}")


out_root.mkdir(parents=True, exist_ok=True)

for _, row in df_all.iterrows():
    item = row["item"]
    tile_name = row["tile"]
    month_str = row["month"].strftime("%Y-%m")

    out_dir = out_root / tile_name / month_str
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nDownloading {tile_name} {month_str} -> {item.id}")
    print(f"  coverage_ratio = {row['coverage_ratio']:.4f}")

    with open(out_dir / f"{item.id}.json", "w") as f:
        json.dump(item.to_dict(), f, indent=2)

    for band in ("vv", "vh"):
        if band not in item.assets:
            continue

        dst = out_dir / f"{item.id}_{band}.tif"
        print(f"  {band.upper()} -> {dst}")
        download_asset_with_fresh_signature(item, band, dst)

print(f"Saved to: {out_root.resolve()}")